# Anchored Sentiment Trajectory Analysis

This notebook computes and visualises an anchored sentiment trajectory over a literary text.

The method represents each text segment with a sentence-transformer model and compares it with two sets of reference examples: positive anchors and negative anchors. Each segment receives a score based on the difference between its average similarity to the positive anchors and its average similarity to the negative anchors.

A positive score means that the segment is closer to the positive anchors. A negative score means that the segment is closer to the negative anchors.

## Workflow

This notebook follows five main steps:

1. Prepare or load a segmented version of the text.
2. Store the segments in a standard dataframe, `segments_df`.
3. Load a sentence-transformer model.
4. Compute anchored sentiment scores for each segment.
5. Visualise the resulting sentiment trajectory.

## Data and copyright note

The full literary text and the anchor sentences are not included in this repository because they contain copyrighted material. To run the notebook locally, provide:

- a prepared local text file or structured source file;
- a local anchor CSV file containing positive and negative anchor sentences;
- a local sentence-transformer model path, or a model name available through `sentence-transformers`.

The notebook is designed so that the preprocessing step can be adapted to different source formats, while the scoring and plotting steps operate on the standard `segments_df` format.

In [ ]:
import sys

import pandas as pd

from pathlib import Path
from sentence_transformers import SentenceTransformer

In [ ]:
PROJECT_ROOT = Path("..").resolve()
SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

## Preprocessing and segmentation

The sentiment-scoring section expects a dataframe named `segments_df` with at least the following columns:

- `segment_text`: the text segment to be scored;
- `chapter`: the chapter or section label aligned with the segment;
- `segment_type`: optional information about the segment type, such as prose, dialogue, verse, or unknown.

Different source formats require different preprocessing rules. Where possible, structured formats such as TEI/XML or HTML should be preferred because paragraph, chapter, and verse boundaries may already be marked. When only plain text is available, source-specific rules must be checked and documented.

The optional parser cells below illustrate possible preprocessing routes. The default workflow in this notebook uses a prepared plain-text file with explicit chapter markers.

The prepared plain-text parser below is used for the local LOTR file in this project. It assumes chapter markers in the form `###CHAPTER:` and a local paragraph-start convention based on leading spaces. These rules are source-specific.

### TEI/XML example (Optional)

This cell is an optional parser template and is not used by the default LOTR plain-text workflow.

In [ ]:
from bs4 import BeautifulSoup

TEI_PATH = Path("data/source.xml")

def segments_from_tei(path: Path) -> pd.DataFrame:
    """
    Extract prose paragraphs and verse blocks from a TEI/XML file.

    This is a template parser. TEI structures vary, so tag names and
    attributes may need to be adapted for a specific corpus.
    """
    with open(path, "r", encoding="utf-8") as file:
        soup = BeautifulSoup(file, "xml")

    rows = []

    for div in soup.find_all("div"):
        chapter_head = div.find("head")
        chapter = chapter_head.get_text(" ", strip=True) if chapter_head else "Unknown"

        for element in div.find_all(["p", "lg"], recursive=True):
            if element.name == "p":
                segment_type = "prose"
                text = element.get_text(" ", strip=True)

            elif element.name == "lg":
                segment_type = "verse"
                lines = [line.get_text(" ", strip=True) for line in element.find_all("l")]
                text = " / ".join(lines) if lines else element.get_text(" ", strip=True)

            else:
                continue

            if text:
                rows.append(
                    {
                        "segment_text": text,
                        "chapter": chapter,
                        "segment_type": segment_type,
                    }
                )

    return pd.DataFrame(rows)

### HTML example (Optional)

This cell is an optional parser template and is not used by the default LOTR plain-text workflow.

In [ ]:
from bs4 import BeautifulSoup

HTML_PATH = Path("data/source.html")

def segments_from_html(path: Path) -> pd.DataFrame:
    """
    Extract paragraphs from an HTML file.

    This works best for HTML/EPUB-derived texts where paragraphs are
    marked with <p> tags. Chapter detection may need to be adapted
    depending on the source.
    """
    with open(path, "r", encoding="utf-8") as file:
        soup = BeautifulSoup(file, "html.parser")

    rows = []
    current_chapter = "Unknown"

    for element in soup.find_all(["h1", "h2", "h3", "p"]):
        if element.name in ["h1", "h2", "h3"]:
            current_chapter = element.get_text(" ", strip=True)

        elif element.name == "p":
            text = element.get_text(" ", strip=True)

            if text:
                rows.append(
                    {
                        "segment_text": text,
                        "chapter": current_chapter,
                        "segment_type": "prose",
                    }
                )

    return pd.DataFrame(rows)

## Prepared plain-text parser used in this notebook

In [ ]:
from literary_nlp.preprocessing import (
    build_segments_df_from_plaintext,
    source_format_diagnostics,
)

### Standard segment dataframe

The segmentation step is standardised into a dataframe named `segments_df`. The later sentiment-scoring cells use this dataframe rather than depending on the specific preprocessing method that produced the segments.

At minimum, `segments_df` contains the segment text and aligned chapter label. A `segment_type` column is included so that future versions can distinguish narration, dialogue, verse, or mixed segments.

In the current plain-text parser, final merged segments are labelled as `mixed_or_unknown`. Future versions may propagate `narration`, `dialogue`, or `verse` labels into the final dataframe.

In [ ]:
CORPUS_PATH = Path("data/source.txt")

segments_df = build_segments_df_from_plaintext(
    CORPUS_PATH,
    min_tok_narr=80,
    max_tok_narr=300,
    min_tok_dial=60,
    max_tok_dial=180,
    max_dialogue_turns=6,
)

segments_df.head()

In [ ]:
diagnostics = source_format_diagnostics(
    CORPUS_PATH,
    segments_df=segments_df,
)

diagnostics

In [ ]:
# Check segment counts by chapter
chapter_segment_counts = (
    segments_df["chapter"]
    .value_counts()
    .sort_index()
)

chapter_segment_counts.head()

## Anchored sentiment scoring

This section scores each segment by comparing it with positive and negative anchor sentences.

For each segment, the model computes:

`sentiment_score = mean_similarity_to_positive_anchors - mean_similarity_to_negative_anchors`

Positive scores indicate that a segment is closer to the positive anchors, while negative scores indicate that it is closer to the negative anchors.

The anchor sentences themselves are not included in the public repository because they are drawn from copyrighted text.

In [ ]:
from literary_nlp.scoring import compute_anchored_sentiment_scores

In [ ]:
# Score corpus segments

MODEL_PATH = Path("../models/tolkien_sentence_transformer_epoch_1")
model = SentenceTransformer(str(MODEL_PATH))

ANCHORS_PATH = Path("data/anchors.csv")

if not ANCHORS_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {ANCHORS_PATH}. "
        "Anchor sentences are not included in the public repository because they contain copyrighted text. "
        "Create a local CSV with columns: sentence_text;label, where label is 'positive' or 'negative'."
    )

anchors_df = pd.read_csv(ANCHORS_PATH, sep=";")

required_cols = {"sentence_text", "label"}
missing = required_cols - set(anchors_df.columns)

if missing:
    raise ValueError(
        f"Missing required columns in anchors CSV: {missing}. "
        f"Present columns: {list(anchors_df.columns)}"
    )

anchors_df["label"] = (
    anchors_df["label"]
    .astype(str)
    .str.strip()
    .str.lower()
)

positive_anchors = anchors_df.loc[
    anchors_df["label"] == "positive",
    "sentence_text",
].tolist()

negative_anchors = anchors_df.loc[
    anchors_df["label"] == "negative",
    "sentence_text",
].tolist()

segments_df["sentiment_score"] = compute_anchored_sentiment_scores(
    model=model,
    segments=segments_df["segment_text"].tolist(),
    positive_anchors=positive_anchors,
    negative_anchors=negative_anchors,
    batch_size=64,
)

segments_df.head()

In [ ]:
# Inspect strongest positive and negative segments

display_cols = ["chapter", "segment_type", "token_count", "sentiment_score", "segment_text"]

most_positive = (
    segments_df
    .sort_values("sentiment_score", ascending=False)
    [display_cols]
    .head(10)
)

most_negative = (
    segments_df
    .sort_values("sentiment_score", ascending=True)
    [display_cols]
    .head(10)
)

most_positive, most_negative

## Visualising the sentiment trajectory

The sentiment trajectory is plotted over the ordered sequence of analysis segments. A rolling mean is used to smooth local variation and make broader narrative movement easier to inspect.

The smoothing window is a visualisation parameter rather than part of the model itself. It should be reported whenever figures generated from this notebook are used in analysis.

In [ ]:
from literary_nlp.plotting import add_rolling_score, plot_trajectory

In [ ]:
# Cell — Prepare smoothed trajectory

WINDOW = 20

plot_df = add_rolling_score(
    segments_df,
    score_col="sentiment_score",
    output_col="smoothed_score",
    window=WINDOW,
)

plot_df.head()

In [ ]:
fig, ax = plot_trajectory(
    plot_df,
    score_col="smoothed_score",
    chapter_col="chapter",
    title=f"Anchored sentiment trajectory (rolling window = {WINDOW})",
    ylabel="Anchored sentiment score",
    positive_label="positive region",
    negative_label="negative region",
    positive_color="steelblue",
    negative_color="indianred",
)